# Sample and null permute portion of every conversation.

Requested by @RickDale

In [6]:
import os
import numpy as np
import pandas as pd
import re
import shutil

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
SERVER_READY_DATA = '../data/server_ready'

OUTPUT_FOLDER = '../data/null-perm'
if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

## Iterate, permute, sample

In [4]:
fs = grab_all_files(SERVER_READY_DATA)
len(fs)

1670

In [6]:
sample_frac = .05

### Everything other than CANDOR

In [7]:
NON_CANDOR = [f for f in fs if (not bool(re.findall(r'CANDOR-\d+-\d+\.parquet', f)))]
len(NON_CANDOR)

14

In [10]:
for f in tqdm(NON_CANDOR):
    table = pq.ParquetFile(f)
    output_ = os.path.join(OUTPUT_FOLDER, f.split('/')[-1]).replace('.parquet', '-null_perm.parquet')

    # print(table.schema.names)
    df = table.read(columns=['corpus', 'conversation_id', 'x_turn_id', 'y_turn_id', 'x_speaker', 'y_speaker', 'x_utterance', 'y_utterance']).to_pandas()

    df['y_utterance'] = df['y_utterance'].sample(frac=1.)

    output_data = []
    groups = df.groupby(by=['corpus', 'conversation_id', 'x_speaker', 'x_turn_id'])
    for labels, dfi in groups:
        output_data += [dfi.sample(n=int(np.ceil(sample_frac * len(dfi))))]

    pd.concat(output_data).to_parquet(output_, engine='fastparquet', compression='snappy')

  0%|          | 0/14 [00:00<?, ?it/s]

### CANDOR Null

In [11]:
CANDOR =[f for f in fs if bool(re.findall(r'CANDOR-\d+-\d+\.parquet', f))]
len(CANDOR)

1656

In [12]:
CANDOR = np.random.choice(
    CANDOR,
    size=(184,9),
    replace=False
)

In [14]:
for i,convoset in enumerate(tqdm(CANDOR)):
    output_ = os.path.join(OUTPUT_FOLDER, 'CANDOR-{}-null_perm.parquet').format(i+1)

    df = pd.concat(
        [pd.read_parquet(f) for f in convoset],
        ignore_index=True
    )
    df['corpus'] = 'CANDOR'

    df['y_utterance'] = df['y_utterance'].sample(frac=1.)

    output_data = []
    groups = df.groupby(by=['corpus', 'conversation_id', 'x_speaker', 'x_turn_id'])
    for labels, dfi in groups:
        output_data += [dfi.sample(n=int(np.ceil(sample_frac * len(dfi))))]

    pd.concat(output_data).to_parquet(output_, engine='fastparquet', compression='snappy')

  0%|          | 0/184 [00:00<?, ?it/s]

## Sort and send

In [4]:
CANDOR = [f for f in os.listdir(OUTPUT_FOLDER) if ('CANDOR-' in f) and (not f.startswith('.'))]
NON_CANDOR = [f for f in os.listdir(OUTPUT_FOLDER) if ('CANDOR-' not in f) and (not f.startswith('.'))]

In [5]:
to_rosen, to_itkin = [], []
to_rosen += np.random.choice(CANDOR, size=(int(len(CANDOR)/2),), replace=False).tolist() + np.random.choice(NON_CANDOR, size=(int(len(NON_CANDOR)/2),), replace=False).tolist()
to_itkin = [f for f in CANDOR if f not in to_rosen] + [f for f in NON_CANDOR if f not in to_rosen]
len(to_itkin), len(to_rosen)

(99, 99)

In [7]:
ROSEN_PATH = os.path.join(OUTPUT_FOLDER, 'to_rosen')
if not os.path.exists(ROSEN_PATH):
    os.mkdir(ROSEN_PATH)

for f in tqdm(to_rosen):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ROSEN_PATH, f)
    )

  0%|          | 0/99 [00:00<?, ?it/s]

In [8]:
ITKIN_PATH = os.path.join(OUTPUT_FOLDER, 'to_itkin')
if not os.path.exists(ITKIN_PATH):
    os.mkdir(ITKIN_PATH)

for f in tqdm(to_itkin):
    shutil.move(
        os.path.join(OUTPUT_FOLDER, f),
        os.path.join(ITKIN_PATH, f)
    )

  0%|          | 0/99 [00:00<?, ?it/s]